# 00 - Setup

1. installs the Python packages (`requirements.txt`),
2. pulls the external reference repos (`external/*`, git submodules) if they're missing,
3. creates the working folders (`data/raw`, `data/processed`, `output`),
4. mounts Google Drive and points `DATA_ROOT` / `OUTPUT_ROOT` at a persistent Drive folder so data and results survive runtime restarts.

## Setup

In [ ]:
# ============================================================
# SETUP: dependencies + external repos + working folders
# ============================================================
import os, subprocess, pathlib

# Run from the repo root
root = pathlib.Path.cwd()
while not (root / "requirements.txt").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
print("Repo root:", root, "\n")

def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

# 1. Python dependencies -------------------------------------------------
sh("pip install -q -r requirements.txt")

# 2. External reference repos --------------------------------------------
externals = {
    "external/Retinal_OCT_Image_Segmentation_via_Deep_Learning": (
        "https://github.com/ZhangHH233/Retinal_OCT_Image_Segmentation_via_Deep_Learning.git",
        "ac6d4c5d263ce03b1bc9235b471a88f9dbb0d994",
    ),
    "external/Public-available-retinal-OCT-datasets": (
        "https://github.com/ZhangHH233/Public-available-retinal-OCT-datasets.git",
        "f50b6a390a1211f06bd34f244c8e53dfc110a266",
    ),
}
for path, (url, commit) in externals.items():
    if any(pathlib.Path(path).glob("*")):
        print(f"OK   {path} (already present)")
        continue
    try:
        sh(f"git submodule update --init -- {path}")
    except subprocess.CalledProcessError:
        sh(f"git clone {url} {path}")
        sh(f"git -C {path} checkout {commit}")

# 3. Working folders (gitignored) ----------------------------------------
# Local scratch on the runtime. 
# If Drive mounted, DATA_ROOT/OUTPUT_ROOT point at Drive instead.
for d in ["data/raw", "data/processed", "output"]:
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
    print(f"OK   {d}/")

print("\nSetup complete — run the verification cell below.")

## Verify environment

In [ ]:
import sys, os
print("Python:", sys.version.split()[0])
!nvidia-smi -L || echo "No GPU detected — set Runtime > Change runtime type > GPU"

import numpy, scipy, skimage, sklearn, PIL, h5py, imageio, cv2, SimpleITK, matplotlib
import torch, torchvision, einops, timm, thop
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("torchvision:", torchvision.__version__, "| SimpleITK:", SimpleITK.Version())

print("\nRepo layout:")
for path in [
    "external/Retinal_OCT_Image_Segmentation_via_Deep_Learning/SOTAS",
    "external/Retinal_OCT_Image_Segmentation_via_Deep_Learning/Metrics",
    "external/Public-available-retinal-OCT-datasets",
    "data/raw", "data/processed", "output",
]:
    print(f"  {'OK     ' if os.path.isdir(path) else 'MISSING'} {path}")

## Mount Google Drive for persistent data + outputs

In [ ]:
# Mount Drive and point the working roots at a persistent Drive folder
# pull to local machine with scripts/pull_from_drive.py
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT  = Path('/content/drive/MyDrive/Segmentation')
DATA_ROOT   = DRIVE_ROOT / 'data'      # raw/ + processed/ datasets
OUTPUT_ROOT = DRIVE_ROOT / 'output'    # metric CSVs, overlays, checkpoints

# Create the Drive-side folders. 
# Keep the 'Segmentation/output' path in sync with scripts/pull_from_drive.py's --drive-base / subpath.
for d in [DATA_ROOT / 'raw', DATA_ROOT / 'processed', OUTPUT_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

print('DATA_ROOT  :', DATA_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)